# Movie store Exploration

Data setup with Sakila film dataset 

In [51]:
import duckdb
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

duckdb_path = "data/sakila.duckdb"
Path(duckdb_path).unlink(missing_ok=True)

with duckdb.connect(duckdb_path) as conn, open("sql/load_sakila.sql") as ingest_script:
    conn.sql(ingest_script.read())

    description = conn.sql("DESC;").df()
    films = conn.sql("FROM film;").df() #films table are now loaded into a pandas dataframe films
    actors = conn.sql("FROM actor;").df() #actors table are now loaded into a pandas dataframe actors
    film_actors = conn.sql("FROM film_actor;").df() #film_actors table are now loaded into a pandas dataframe film_actors
    inventory = conn.sql("SELECT * FROM inventory;").df()# Load inventory table
    rental = conn.sql("SELECT * FROM rental;").df()# Load rental table
    payment = conn.sql("SELECT * FROM payment;").df()# Load payment table
    customer = conn.sql("SELECT * FROM customer;").df()# Load customer table

films.head(3)

ModuleNotFoundError: No module named 'matplotlib'

Which movies are longer than 3 hours (180 minutes), lets explore the title and its length

In [ ]:

films['length'].head(3)

0    86
1    48
2    50
Name: length, dtype: int64

In [ ]:
long_films = films[films["length"] > 180]
long_films.head()

,film_id,title,description,release_year,language_id,original_language_id,rental_duration,rental_rate,length,replacement_cost,rating,special_features,last_update
23,24,ANALYZE HOOSIERS,A Thoughtful Display of a Explorer And a Pastr...,2006,1,<NA>,6,2.99,181,19.99,R,"Trailers,Behind the Scenes",2021-03-06 15:52:00
49,50,BAKED CLEOPATRA,A Stunning Drama of a Forensic Psychologist An...,2006,1,<NA>,3,2.99,182,20.99,G,"Commentaries,Behind the Scenes",2021-03-06 15:52:01
127,128,CATCH AMISTAD,A Boring Reflection of a Lumberjack And a Femi...,2006,1,<NA>,7,0.99,183,10.99,G,"Trailers,Behind the Scenes",2021-03-06 15:52:01
140,141,CHICAGO NORTH,A Fateful Yarn of a Mad Cow And a Waitress who...,2006,1,<NA>,6,4.99,185,11.99,PG-13,"Deleted Scenes,Behind the Scenes",2021-03-06 15:52:01
179,180,CONSPIRACY SPIRIT,A Awe-Inspiring Story of a Student And a Frisb...,2006,1,<NA>,4,2.99,184,27.99,PG-13,"Trailers,Commentaries",2021-03-06 15:52:02


In [ ]:
result = long_films[["title", "length"]]
result = result.sort_values("length", ascending=False)
print(result)

                  title  length
689        POND SEATTLE     185
990        WORST BANGER     185
140       CHICAGO NORTH     185
181      CONTROL ANTHEM     185
816  SOLDIERS EVOLUTION     185
211      DARN FORRESTER     185
348         GANGS PRIDE     185
608       MUSCLE BRIGHT     185
425           HOME PITY     185
871   SWEET BROTHERHOOD     185
819      SONS INTERVIEW     184
812     SMOOCHY CONTROL     184
498      KING EVOLUTION     184
596     MOONWALKER FOOL     184
820      SORORITY QUEEN     184
197    CRYSTAL BREAKING     184
179   CONSPIRACY SPIRIT     184
885      THEORY MERMAID     184
972           WIFE TURN     183
766       SCALAWAG DUCK     183
995      YOUNG LANGUAGE     183
127       CATCH AMISTAD     183
339      FRONTIER CABIN     183
49      BAKED CLEOPATRA     182
720          REDS POCUS     182
764         SATURN NAME     182
590       MONSOON CAUSE     182
773      SEARCHERS WAIT     182
718       RECORDS ZORRO     182
434     HOTEL HAPPINESS     181
973     

Films with LOVE in their title

In [ ]:
result = films.loc[
    films["title"].str.contains("love", case=False, na=False),
    ["title", "rating", "length", "description"]
]
print(result.sort_values("title"))

                  title rating  length  \
373       GRAFFITI LOVE     PG     117   
447          IDAHO LOVE  PG-13     172   
448      IDENTITY LOVER  PG-13     119   
457         INDIAN LOVE  NC-17     135   
510       LAWRENCE LOVE  NC-17     175   
534       LOVE SUICIDES      R     181   
535       LOVELY JINGLE     PG      65   
536        LOVER TRUMAN      G      75   
537    LOVERBOY ATTACKS  PG-13     162   
851  STRANGELOVE DESIRE  NC-17     103   

                                           description  
373  A Unbelieveable Epistle of a Sumo Wrestler And...  
447  A Fast-Paced Drama of a Student And a Crocodil...  
448  A Boring Tale of a Composer And a Mad Cow who ...  
457  A Insightful Saga of a Mad Scientist And a Mad...  
510  A Fanciful Yarn of a Database Administrator An...  
534  A Brilliant Panorama of a Hunter And a Explore...  
535  A Fanciful Yarn of a Crocodile And a Forensic ...  
536  A Emotional Yarn of a Robot And a Boy who must...  
537  A Boring Story of a

## Statistics 
Descriptive stats of the length of films

In [ ]:
conn = duckdb.connect()

query = """
SELECT
    MIN(length)   AS shortest_length,
    AVG(length)   AS average_length,
    MEDIAN(length) AS median_length,
    MAX(length)   AS longest_length
FROM films;
"""

conn.execute(query).df()


,shortest_length,average_length,median_length,longest_length
0,46,115.272,114.0,185


Rental rate

In [ ]:
conn = duckdb.connect()

query = """
SELECT
    title,
    rental_rate,
    rental_duration,
    rental_rate / rental_duration AS cost_per_day
FROM films
WHERE rental_duration > 0
ORDER BY cost_per_day DESC
LIMIT 10;
"""

conn.execute(query).df()


,title,rental_rate,rental_duration,cost_per_day
0,BACKLASH UNDEFEATED,4.99,3,1.663333
1,BILKO ANONYMOUS,4.99,3,1.663333
2,BEAST HUNCHBACK,4.99,3,1.663333
3,AUTUMN CROW,4.99,3,1.663333
4,ACE GOLDFINGER,4.99,3,1.663333
5,CARIBBEAN LIBERTY,4.99,3,1.663333
6,BEHAVIOR RUNAWAY,4.99,3,1.663333
7,CASPER DRAGONFLY,4.99,3,1.663333
8,AMERICAN CIRCUS,4.99,3,1.663333
9,CASUALTIES ENCINO,4.99,3,1.663333


Most popular actors

In [ ]:
conn = duckdb.connect()

query = """
SELECT
    a.actor_id,
    a.first_name,
    a.last_name,
    COUNT(fa.film_id) AS movie_count
FROM actors a
JOIN film_actors fa
    ON a.actor_id = fa.actor_id
JOIN films f
    ON fa.film_id = f.film_id
GROUP BY a.actor_id, a.first_name, a.last_name
ORDER BY movie_count DESC
LIMIT 10;
"""

conn.execute(query).df()

,actor_id,first_name,last_name,movie_count
0,107.0,GINA,DEGENERES,42
1,102.0,WALTER,TORN,41
2,198.0,MARY,KEITEL,40
3,181.0,MATTHEW,CARREY,39
4,23.0,SANDRA,KILMER,37
5,81.0,SCARLETT,DAMON,36
6,60.0,HENRY,BERRY,35
7,106.0,GROUCHO,DUNST,35
8,144.0,ANGELA,WITHERSPOON,35
9,37.0,VAL,BOLGER,35


Which are the most popular years of film production - i.e which years where most films released.

In [ ]:
conn = duckdb.connect()

query = """
SELECT
    release_year,
    COUNT(*) AS num_films
FROM films
GROUP BY release_year
ORDER BY num_films DESC
LIMIT 10;
"""

conn.execute(query).df()

,release_year,num_films
0,2006,1000


Which films are the most profitable.

In [ ]:
conn = duckdb.connect()

query = """
SELECT
    f.title,
    SUM(p.amount) AS total_revenue
FROM films f
JOIN inventory i
    ON f.film_id = i.film_id
JOIN rental r
    ON i.inventory_id = r.inventory_id
JOIN payment p
    ON r.rental_id = p.rental_id
GROUP BY f.film_id, f.title
ORDER BY total_revenue DESC
LIMIT 10;
"""

conn.execute(query).df()


,title,total_revenue
0,TELEGRAPH VOYAGE,231.73
1,WIFE TURN,223.69
2,ZORRO ARK,214.69
3,GOODFELLAS SALUTE,209.69
4,SATURDAY LAMBS,204.72
5,TITANS JERK,201.71
6,TORQUE BOUND,198.72
7,HARRY IDAHO,195.70
8,INNOCENT USUAL,191.74
9,HUSTLER PARTY,190.78


## Graphs

In [ ]:
conn = duckdb.connect()

query = """
SELECT
    c.customer_id,
    c.first_name || ' ' || c.last_name AS customer_name,
    SUM(p.amount) AS total_spend
FROM customer c
JOIN payment p
    ON c.customer_id = p.customer_id
GROUP BY c.customer_id, customer_name
ORDER BY total_spend DESC
LIMIT 5;
"""

top_customers = conn.sql(query).df()
print(top_customers)


   customer_id   customer_name  total_spend
0          526       KARL SEAL       221.55
1          148    ELEANOR HUNT       216.54
2          144      CLARA SHAW       195.58
3          178   MARION SNYDER       194.61
4          137  RHONDA KENNEDY       194.61


In [ ]:
plt.figure(figsize=(10,6))
plt.bar(top_customers['customer_name'], top_customers['total_spend'], color='skyblue')
plt.xlabel('Customer')
plt.ylabel('Total Spend ($)')
plt.title('Top 5 Customers by Total Spend')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


Explorative data analysis 

In [ ]:
dfs = {}

with duckdb.connect(duckdb_path) as conn:
    for name in description["name"]:
        dfs[name] = conn.sql(f"FROM {name};").df()

dfs.keys()

dict_keys(['actor', 'address', 'category', 'city', 'country', 'customer', 'customer_list', 'film', 'film_actor', 'film_category', 'film_list', 'film_text', 'inventory', 'language', 'payment', 'rental', 'sales_by_film_category', 'sales_by_store', 'staff', 'staff_list', 'store'])

In [ ]:
film_names = ("film", "film_actor", "film_category", "actor", "category")

for film_name in film_names:
    duckdb.register(film_name, dfs[film_name])

duckdb.sql("desc;").df()

,database,schema,name,column_names,column_types,temporary
0,temp,main,actor,"[actor_id, first_name, last_name, last_update]","[DOUBLE, VARCHAR, VARCHAR, TIMESTAMP]",True
1,temp,main,category,"[category_id, name, last_update]","[BIGINT, VARCHAR, TIMESTAMP]",True
2,temp,main,film,"[film_id, title, description, release_year, la...","[BIGINT, VARCHAR, VARCHAR, VARCHAR, BIGINT, BI...",True
3,temp,main,film_actor,"[actor_id, film_id, last_update]","[BIGINT, BIGINT, TIMESTAMP]",True
4,temp,main,film_category,"[film_id, category_id, last_update]","[BIGINT, BIGINT, TIMESTAMP]",True


In [ ]:
films_joined = duckdb.sql("""
    SELECT
        a.first_name || ' ' || a.last_name AS actor,
        a.actor_id::INT AS actor_id,
        f.title,
        f.description,
        f.release_year,
        f.rental_duration,
        f.rating,
        c.name AS category
    FROM film f
        LEFT JOIN film_actor fa ON f.film_id = fa.film_id
        LEFT JOIN actor a ON a.actor_id = fa.actor_id
        LEFT JOIN film_category fc ON fc.film_id = f.film_id      
        LEFT JOIN category c ON fc.category_id = c.category_id      
""").df()

films_joined.head(2)

,actor,actor_id,title,description,release_year,rental_duration,rating,category
0,PENELOPE GUINESS,1,ACADEMY DINOSAUR,A Epic Drama of a Feminist And a Mad Scientist...,2006,6,PG,Documentary
1,PENELOPE GUINESS,1,ANACONDA CONFESSIONS,A Lacklusture Display of a Dentist And a Denti...,2006,3,R,Animation
